# LightGBM 4 lá với mã hóa mục tiêu

Bài thực hành phân loại lỗi sản phẩm trên dữ liệu tự sinh. Các mô hình được huấn luyện
từ đầu. Notebook chạy trên CPU và tự tạo dữ liệu để có thể chạy độc lập trên Kaggle.

## 1. Tạo dữ liệu lỗi sản phẩm giả lập

Mỗi hàng mô phỏng một sản phẩm. Các cột mô tả cảm biến, dây chuyền, nhà cung cấp
và ca làm việc. Nhãn `defect` được rút theo xác suất phụ thuộc vào các yếu tố đó.
Hạt giống ngẫu nhiên (random seed) giữ dữ liệu giống nhau trong cả năm notebook.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 2026
N_ROWS = 20_000
rng = np.random.default_rng(SEED)

line = rng.integers(0, 8, N_ROWS)
supplier = rng.integers(0, 16, N_ROWS)
shift = rng.choice(['day', 'swing', 'night'], N_ROWS, p=[0.45, 0.35, 0.20])
temperature = rng.normal(74, 7, N_ROWS)
pressure = rng.normal(102, 9, N_ROWS)
vibration = rng.lognormal(-0.2, 0.48, N_ROWS)
feed_speed = rng.normal(55, 8, N_ROWS)
tool_age = rng.gamma(2, 11, N_ROWS)
line_effect = rng.normal(0, 0.55, 8)
supplier_effect = rng.normal(0, 0.55, 16)

log_odds = (
    -4.65
    + 0.72 * np.log1p(vibration / 0.5)
    + 0.46 * (temperature - 74) / 7
    + 0.44 * (tool_age - 22) / 16
    + 0.36 * np.abs(pressure - 102) / 9
    + 0.18 * (feed_speed - 55) / 8
    + 0.80 * ((vibration > 1.3) & (temperature > 80))
    + 0.48 * (shift == 'night')
    + line_effect[line]
    + supplier_effect[supplier]
)
probability = 1 / (1 + np.exp(-log_odds))
frame = pd.DataFrame({
    'temperature': temperature,
    'pressure': pressure,
    'vibration': vibration,
    'feed_speed': feed_speed,
    'tool_age': tool_age,
    'production_line': [f'line_{i:02}' for i in line],
    'supplier': [f'supplier_{i:02}' for i in supplier],
    'shift': shift,
    'defect': rng.binomial(1, probability),
})
NUMERIC_COLUMNS = ['temperature', 'pressure', 'vibration', 'feed_speed', 'tool_age']
CATEGORICAL_COLUMNS = ['production_line', 'supplier', 'shift']

print(frame.head(4).round(3).to_string(index=False))
print(f"Số mẫu: {len(frame):,} | Tỷ lệ lỗi: {frame['defect'].mean():.2%}")

 temperature  pressure  vibration  feed_speed  tool_age production_line    supplier shift  defect
      73.039   109.978      1.137      55.768    50.912         line_06 supplier_13 swing       1
      67.415   110.243      0.682      54.665    12.969         line_01 supplier_14   day       0
      65.495   110.196      1.107      61.027     6.746         line_00 supplier_04   day       0
      82.278    95.993      0.968      58.438    68.024         line_05 supplier_12 night       0
Số mẫu: 20,000 | Tỷ lệ lỗi: 4.37%


## 2. Chia tập huấn luyện, xác thực và kiểm tra

Chia phân tầng (stratified split) giúp giữ tỷ lệ sản phẩm lỗi gần nhau ở ba tập.
Tập huấn luyện dùng để học mô hình và bước tiền xử lý. Tập xác thực dùng để theo dõi
chất lượng hoặc chọn trọng số. Tập kiểm tra được giữ riêng cho lần chấm cuối.

In [2]:
X = frame[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS]
y = frame['defect'].to_numpy(dtype=np.int64)
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_hold, y_hold, test_size=0.50, random_state=SEED, stratify=y_hold,
)
split_hash = hashlib.sha256(
    X_test.index.to_numpy(dtype='<i8').tobytes() + y_test.astype('<i8').tobytes()
).hexdigest()[:16]
for name, labels in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f"{name:10s}  {len(labels):6,d} mẫu | {labels.sum():4,d} lỗi ({labels.mean():.2%})")
print(f'Mã kiểm tra tập test: {split_hash}')

Train       14,000 mẫu |  611 lỗi (4.36%)
Validation   3,000 mẫu |  131 lỗi (4.37%)
Test         3,000 mẫu |  131 lỗi (4.37%)
Mã kiểm tra tập test: 94efc544e1e83429


### Mã hóa biến phân loại theo nhãn

Mã hóa theo nhãn (target encoding) thay mỗi giá trị phân loại bằng tỷ lệ lỗi ước lượng. Tập huấn luyện chia thành 5 phần có tỷ lệ nhãn xấp xỉ nhau. Mỗi phần dùng tỷ lệ tính từ 4 phần còn lại.

Các nhóm ít mẫu được kéo về tỷ lệ lỗi chung với trọng số tương đương 20 mẫu. Tập kiểm định và tập kiểm thử dùng bảng mã từ toàn bộ tập huấn luyện. Giá trị chưa gặp nhận tỷ lệ lỗi chung. Các cột số được giữ nguyên ở bước này.

In [3]:
from sklearn.model_selection import StratifiedKFold

SMOOTHING = 20.0


def smoothed_rates(categories, labels):
    prior = float(np.mean(labels))
    observations = pd.DataFrame(
        {"category": categories.to_numpy(), "label": labels}
    )
    stats = observations.groupby("category", dropna=False)["label"].agg(
        ["sum", "count"]
    )
    rates = (stats["sum"] + SMOOTHING * prior) / (
        stats["count"] + SMOOTHING
    )
    return rates, prior


def apply_rates(categories, rates, prior):
    return categories.map(rates).fillna(prior).to_numpy(dtype=float)


fold_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(fold_splitter.split(X_train, y_train))
train_encoded = X_train[NUMERIC_COLUMNS].copy()
val_encoded = X_val[NUMERIC_COLUMNS].copy()
test_encoded = X_test[NUMERIC_COLUMNS].copy()

for column in CATEGORICAL_COLUMNS:
    categories = X_train[column]
    out_of_fold = np.empty(len(X_train), dtype=float)
    for fit_idx, held_out_idx in folds:
        rates, prior = smoothed_rates(
            categories.iloc[fit_idx], y_train[fit_idx]
        )
        out_of_fold[held_out_idx] = apply_rates(
            categories.iloc[held_out_idx], rates, prior
        )
    train_encoded[column] = out_of_fold

    rates, prior = smoothed_rates(categories, y_train)
    val_encoded[column] = apply_rates(X_val[column], rates, prior)
    test_encoded[column] = apply_rates(X_test[column], rates, prior)

### LightGBM

LightGBM huấn luyện 220 cây liên tiếp. Mỗi cây có tối đa 4 lá và độ sâu tối đa 2 để giới hạn độ phức tạp. Thời gian ghi lại cho bước gọi fit, tính bằng giây. Bước mã hóa và dự đoán được thực hiện riêng.

In [4]:
from time import perf_counter

from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=220,
    learning_rate=0.05,
    num_leaves=4,
    max_depth=2,
    device_type="cpu",
    n_jobs=2,
    random_state=SEED,
    verbosity=-1,
)
started_at = perf_counter()
model.fit(train_encoded, y_train)
fit_seconds = perf_counter() - started_at

val_probs = model.predict_proba(val_encoded)[:, 1]
test_probs = model.predict_proba(test_encoded)[:, 1]

## Đánh giá và đọc kết quả

ROC AUC đo khả năng xếp sản phẩm lỗi cao hơn sản phẩm đạt. Average Precision
(AP) tóm tắt độ chính xác của danh sách cảnh báo khi tăng số mẫu được phát hiện.
Tỷ lệ lỗi của tập test là mốc AP để tham khảo với cách xếp hạng ngẫu nhiên.

In [5]:
MODEL_SLUG = 'lightgbm-te'
for name, labels, scores in [('validation', y_val, val_probs), ('test', y_test, test_probs)]:
    assert scores.shape == labels.shape, f'Sai kích thước dự đoán trên {name}'
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()

metrics = {
    'model': MODEL_SLUG,
    'seed': SEED,
    'train_rows': len(y_train),
    'validation_rows': len(y_val),
    'test_rows': len(y_test),
    'test_positive_count': int(y_test.sum()),
    'test_positive_rate': float(y_test.mean()),
    'test_split_hash': split_hash,
    'validation_roc_auc': float(roc_auc_score(y_val, val_probs)),
    'test_roc_auc': float(roc_auc_score(y_test, test_probs)),
    'test_average_precision': float(average_precision_score(y_test, test_probs)),
    'fit_seconds': float(fit_seconds),
}
on_kaggle = Path('/kaggle/working').is_dir()
output_dir = Path('/kaggle/working') if on_kaggle else Path('results')
output_dir.mkdir(exist_ok=True)
output_file = output_dir / f'{MODEL_SLUG}-metrics.json'
output_file.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print(f"ROC AUC validation: {metrics['validation_roc_auc']:.4f}")
print(f"ROC AUC test:       {metrics['test_roc_auc']:.4f}")
print(f"Average Precision:  {metrics['test_average_precision']:.4f}")
print(f"Tỷ lệ lỗi tập test: {metrics['test_positive_rate']:.2%}")
print(f"Thời gian fit:      {metrics['fit_seconds']:.2f} giây")
print(f"Đã lưu kết quả:    {output_file}")

ROC AUC validation: 0.7412
ROC AUC test:       0.7687
Average Precision:  0.1699
Tỷ lệ lỗi tập test: 4.37%
Thời gian fit:      0.06 giây
Đã lưu kết quả:    results\lightgbm-te-metrics.json
